In [1]:
import pandas as pd
import numpy as np

# Load dataset
file_path = 'Superstore Dataset Source (1).xlsx'
df_raw = pd.read_excel(file_path, sheet_name='Orders')

print("Raw Dataset Loaded Successfully.")
print(f"Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

Raw Dataset Loaded Successfully.
Shape: 9994 rows, 21 columns


In [2]:
# Create Initial Data Quality Audit Report
initial_audit = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Null Count': df_raw.isnull().sum(),
    'Null Percentage (%)': (df_raw.isnull().sum() / len(df_raw)) * 100,
    'Unique Values': df_raw.nunique()
})

duplicate_rows_before = df_raw.duplicated().sum()

print("=== DATA QUALITY AUDIT (BEFORE CLEANING) ===")
print(f"Duplicate Rows Count: {duplicate_rows_before}")
print("\nColumn Summary:")
print(initial_audit)

=== DATA QUALITY AUDIT (BEFORE CLEANING) ===
Duplicate Rows Count: 0

Column Summary:
                    Data Type  Null Count  Null Percentage (%)  Unique Values
Row ID                  int64           0                  0.0           9994
Order ID                  str           0                  0.0           5009
Order Date     datetime64[us]           0                  0.0           1237
Ship Date      datetime64[us]           0                  0.0           1334
Ship Mode                 str           0                  0.0              4
Customer ID               str           0                  0.0            793
Customer Name             str           0                  0.0            793
Segment                   str           0                  0.0              3
Country                   str           0                  0.0              1
City                      str           0                  0.0            531
State                     str           0               

Data Quality Audit Findings:

Null Values: Currently 0 null values across primary columns in this source.

Duplicates: 0 exact duplicate rows found in raw data.

Data Types: Order Date and Ship Date are correctly formatted as datetimes; numerical columns (Sales, Quantity, Discount, Profit) are floating/integer types. Postal Code is stored as an integer and needs string conversion to prevent leading zeros from dropping.

In [3]:
df_cleaned = df_raw.copy()

# Step 1: Remove Duplicate Rows (if any)
initial_count = len(df_cleaned)
df_cleaned.drop_duplicates(inplace=True)
duplicates_removed = initial_count - len(df_cleaned)

# Step 2: Fix Data Types (Convert Postal Code and IDs to string)
df_cleaned['Postal Code'] = df_cleaned['Postal Code'].astype(str).str.zfill(5)
df_cleaned['Customer ID'] = df_cleaned['Customer ID'].astype(str)
df_cleaned['Order ID'] = df_cleaned['Order ID'].astype(str)

# Step 3: Handle Inconsistent Formatting / Whitespaces in Categorical Features
categorical_cols = ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Region', 'Category', 'Sub-Category']
for col in categorical_cols:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip().str.title()

# Step 4: Handle Outliers in Numeric Columns using IQR Method
num_cols = ['Sales', 'Profit', 'Discount', 'Quantity']
outlier_summary = {}

for col in num_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_cleaned[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound)]
    outlier_summary[col] = len(outliers)

print(f"Duplicates Removed: {duplicates_removed}")
print("\nOutliers Detected per Column (IQR Method):")
for col, count in outlier_summary.items():
    print(f" - {col}: {count} potential outliers")

Duplicates Removed: 0

Outliers Detected per Column (IQR Method):
 - Sales: 1167 potential outliers
 - Profit: 1881 potential outliers
 - Discount: 856 potential outliers
 - Quantity: 170 potential outliers


In [ ]:
Data Cleaning Decisions & Justifications:

Postal Code Standardisation: Converted Postal Code to 5-digit zero-padded strings (zfill(5)) to preserve valid U.S. ZIP codes with leading zeros (e.g., 01841).

Text Normalization: Stripped whitespace and applied Title Case formatting to all categorical columns to prevent duplicate grouping issues (e.g., "second class" vs "Second Class").

Outlier Strategy: Retained numerical outliers in Sales and Profit because high-value transaction orders and negative profits are legitimate retail occurrences, not data entry errors.